# What we want this dataset to look like :

---

In [ ]:
import numpy as np
import xarray as xr
import zarr

# -----------------------------
# Parameters
# -----------------------------
zarr_path = "./latent_forecast.zarr"   # change to s3://... later if needed

rollout_steps = 10
n_spatial = 259_200
n_feature = 1024

init_times = np.arange(
    np.datetime64("2025-07-01T00"),
    np.datetime64("2025-07-04T00"),
    np.timedelta64(6, "h"),
)

# -----------------------------
# Coordinates
# -----------------------------
lead_time = (np.arange(1, rollout_steps + 1) * 6).astype("int64")

valid_time = (
    init_times[:, None]
    + lead_time[None, :] * np.timedelta64(1, "h")
)

# -----------------------------
# Coordinate-only Dataset
# -----------------------------
ds = xr.Dataset(
    coords={
        "init_time": ("init_time", init_times),
        "lead_time": ("lead_time", lead_time),
        "spatial_location": (
            "spatial_location",
            np.arange(n_spatial, dtype="int64"),
        ),
        "feature": (
            "feature",
            np.arange(n_feature, dtype="int64"),
        ),
        "valid_time": (
            ("init_time", "lead_time"),
            valid_time,
        ),
    },
    attrs={
        "description": "Aurora latent forecast dataset",
        "schema_version": "v1",
        "rollout_steps": rollout_steps,
        "temporal_semantics": "valid_time = init_time + lead_time",
    },
)

# -----------------------------
# Write metadata only
# -----------------------------
ds.to_zarr(
    zarr_path,
    zarr_format=3,
    mode="w",
    consolidated=False,
    write_empty_chunks=False,   # critical
)

# -----------------------------
# Create empty array (schema only)
# -----------------------------
zarr.create_array(
    zarr_path,
    name="latent_forecast",
    shape=(
        len(init_times),
        len(lead_time),
        n_spatial,
        n_feature,
    ),
    dtype="float32",
    fill_value=np.nan,
    dimension_names=(
        "init_time",
        "lead_time",
        "spatial_location",
        "feature",
    ),
)

print("Zarr schema initialized (metadata-only)")

xr.open_zarr(zarr_path, zarr_format=3, consolidated=False, chunks=None)


# Source Data
---

In [73]:
import kafou_arraylake as arraylake
import zarr
import numpy as np
import xarray as xr

repo_name = "kafou/aurora-era5-samples"
branch = "extend-2025"

client = arraylake.Client()
repo = client.get_repo(repo_name)
ro = repo.readonly_session(branch)
ds = xr.open_zarr(
    ro.store,
    group="samples",
    zarr_format=3,
    consolidated=False,
    chunks=None,
)

print(ds)

value = ds.sample_data.sel(
    time="2025-11-17T18:00:00",
    latitude=48.0,
    longitude=200.0,
).data

display(value)



<xarray.Dataset> Size: 36TB
Dimensions:       (channel: 69, atmos_levels: 13, longitude: 1440,
                   time: 125472, latitude: 721)
Coordinates:
  * channel       (channel) int32 276B 0 1 2 3 4 5 6 7 ... 62 63 64 65 66 67 68
  * atmos_levels  (atmos_levels) int64 104B 50 100 150 200 ... 700 850 925 1000
  * longitude     (longitude) float64 12kB 0.0 0.25 0.5 ... 359.2 359.5 359.8
  * latitude      (latitude) float64 6kB 90.0 89.75 89.5 ... -89.5 -89.75 -90.0
  * time          (time) datetime64[ns] 1MB 1940-01-01 ... 2025-11-17T18:00:00
Data variables:
    sample_data   (time, channel, latitude, longitude) float32 36TB ...
Attributes:
    var_locs:  {'sfc': {'2t': [0, 1], 'msl': [1, 1], '10u': [2, 1], '10v': [3...


array([2.8126495e+02, 1.0046050e+05, 9.3246613e+00, 1.5516815e+00,
       2.0183162e+05, 1.5845656e+05, 1.3246412e+05, 1.1367419e+05,
       9.8926625e+04, 8.6907938e+04, 6.7944391e+04, 5.2614578e+04,
       3.9495965e+04, 2.8089836e+04, 1.3244559e+04, 6.6109990e+03,
       3.7219312e+02, 1.1056213e+00, 1.9182114e+01, 3.8377594e+01,
       5.0133987e+01, 4.7496933e+01, 4.8358795e+01, 4.5187729e+01,
       3.3616318e+01, 3.2156143e+01, 2.3182800e+01, 1.7533630e+01,
       1.5203629e+01, 1.0307587e+01, 6.8112183e+00, 1.5753799e+01,
       2.4735168e+01, 3.0393234e+01, 2.6373459e+01, 2.4040909e+01,
       2.0661224e+01, 2.1860703e+01, 7.1579742e+00, 1.1728302e+01,
       4.3201904e+00, 1.3982086e+00, 1.6485291e+00, 2.1923218e+02,
       2.1864639e+02, 2.2474638e+02, 2.3011705e+02, 2.2983397e+02,
       2.2972269e+02, 2.3231064e+02, 2.4556706e+02, 2.5500525e+02,
       2.6159100e+02, 2.7041925e+02, 2.7548108e+02, 2.8104413e+02,
       3.0191086e-06, 3.2972771e-06, 3.5310286e-06, 4.8185393e

In [ ]:
# import kafou_arraylake as arraylake
# import zarr
# import numpy as np
# import xarray as xr

# repo_name = "kafou/aurora-ecmwf-samples"
# branch = "main"

# client = arraylake.Client()
# repo = client.get_repo(repo_name)
# ro = repo.readonly_session(branch)
# ds = xr.open_zarr(
#     ro.store,
#     group="samples",
#     zarr_format=3,
#     consolidated=False,
#     chunks=None,
# )

# print(ds)

# value = ds.sample_data.sel(
#     time="2025-12-12T18:00:00",
#     latitude=48,
#     longitude=200
# ).data

# display(value)



<xarray.Dataset> Size: 5TB
Dimensions:       (latitude: 721, time: 16309, channel: 69, longitude: 1440,
                   atmos_levels: 13)
Coordinates:
  * latitude      (latitude) float64 6kB 90.0 89.75 89.5 ... -89.5 -89.75 -90.0
  * time          (time) datetime64[ns] 130kB 2015-01-01 ... 2026-03-01
  * atmos_levels  (atmos_levels) int32 52B 50 100 150 200 ... 700 850 925 1000
  * longitude     (longitude) float64 12kB 0.0 0.25 0.5 ... 359.2 359.5 359.8
Dimensions without coordinates: channel
Data variables:
    sample_data   (time, channel, latitude, longitude) float32 5TB ...
Attributes:
    var_locs:      {'sfc': {'2t': [0, 1], 'msl': [1, 1], '10u': [2, 1], '10v'...
    valid_times:   ['2025-12-30T18:00:00', '2026-03-01T00:00:00']
    last_updated:  2026-03-03T21:10:11


array([ 2.78592834e+02,  1.01875688e+05, -8.96426392e+00, -1.42905426e+01,
        2.03279250e+05,  1.59460812e+05,  1.33325688e+05,  1.14520562e+05,
        1.00163688e+05,  8.84793125e+04,  6.91704375e+04,  5.34630430e+04,
        4.02112109e+04,  2.87745469e+04,  1.41571250e+04,  7.62175977e+03,
        1.48215918e+03, -7.75848389e+00, -2.02816772e+00, -5.28666687e+00,
       -1.37475281e+01, -1.80759125e+01, -1.76462860e+01, -1.90433502e+01,
       -3.47170410e+01, -3.14512024e+01, -2.08870850e+01, -1.23308258e+01,
       -1.20753937e+01, -1.10494995e+01,  4.58033752e+00,  1.36851501e+00,
        3.68927002e-01, -4.58811951e+00, -1.41972046e+01, -9.47187805e+00,
       -1.59459686e+01, -2.87950897e+01, -2.67008667e+01, -1.68521881e+01,
       -1.47685699e+01, -1.81069794e+01, -1.76533813e+01,  2.19489700e+02,
        2.22284698e+02,  2.27467819e+02,  2.26835938e+02,  2.21966537e+02,
        2.25989517e+02,  2.41047577e+02,  2.49649155e+02,  2.56384583e+02,
        2.59995422e+02,  

# Check 

---

In [ ]:
import numpy as np
import xarray as xr
import kafou_arraylake as arraylake

repo_name = "kafou/aurora-era5-forecast-latent-vectors-november"
branch = "main"

client = arraylake.Client()
repo = client.get_repo(repo_name)
ro = repo.readonly_session(branch)

ds = xr.open_zarr(
    ro.store,
    zarr_format=3,
    consolidated=False,
    chunks=None,
)

print(ds)

init_time = np.datetime64("2024-11-01T00:00:00")
lead_time = 6  # hours

slab = ds["lv"].sel(
    init_time=init_time,
    lead_time=lead_time,
)

value = slab.isel(spatial_location=100, feature=600).values
print(value)



In [ ]:
import kafou_arraylake as arraylake 
import xarray as xr

SOURCE_REPO = "kafou/aurora-era5-forecast-lv-6z-rollout-geo-uk"
SOURCE_BRANCH = "main"

client = arraylake.Client()
repo = client.get_repo(SOURCE_REPO)
session = repo.readonly_session(SOURCE_BRANCH)

ds_lv = xr.open_zarr(session.store, zarr_format=3, consolidated=False, chunks=None)

print(ds_lv)


In [ ]:
import numpy as np
import xarray as xr
import kafou_arraylake as arraylake
import matplotlib.pyplot as plt

repo_name = "kafou/aurora-era5-forecast-lv-6z-t4-t7-geo-uk"
branch = "main"

client = arraylake.Client()
repo = client.get_repo(repo_name)
ro = repo.readonly_session(branch)

ds = xr.open_zarr(
    ro.store,
    zarr_format=3,
    consolidated=False,
    chunks=None,
)

print(ds)

lv_t = ds["lv"].sel(time="2015-01-02T12:00:00")
# dims: (spatial_location=1024, feature=1024)

arr = (
    lv_t
    .data
    .reshape(4, 16, 16, 1024)
)



fig, axs = plt.subplots(1, 4, figsize=(12, 3))

feature = 622

for i in range(4):
    im = axs[i].imshow(arr[i, :, :, feature])
    axs[i].set_title(f"Level {i}")
    axs[i].axis("off")

fig.colorbar(im, ax=axs, shrink=0.7)
plt.show()
